# 🏛️ NeuralOps Quickstart: 1-Line Surgical Attention Replacement & Micro-Healing

Welcome to **NeuralOps v1.3.7**! This tutorial provides an end-to-end, reproducible guide for:
1. **Architectural Inspection**: Inspecting mathematical equations, 4-stage algebraic decompositions ($P \to \phi \to M \to E$), and complexity guarantees using `neuralops.inspect_architecture`.
2. **Direct Attention Instantiation**: Instantiating subquadratic linear attention modules directly with `neuralops.create_attention`.
3. **1-Line Surgical Attention Replacement**: Replacing quadratic $O(T^2)$ self-attention layers in pretrained Hugging Face models (e.g., GPT-2) with subquadratic $O(T \cdot d^2)$ recurrent attention (`gated_delta_net`) using `neuralops.replace_attention`.
4. **Subspace Micro-Healing & Ridge Calibration**: Recovering representation geometry via closed-form Ridge regression and layerwise gradient micro-healing with `neuralops.heal_replaced_model`.
5. **Bit-Exact Surgical Checkpointing & Ledger Auditing**: Persisting surgical checkpoints with bit-exact manifests via `neuralops.save_surgical_checkpoint` and querying execution telemetry in `neuralops.get_default_ledger`.

---
### Mathematical Formulation: Gated Delta Net

Standard softmax attention computes:
$$\text{Attn}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V \quad \left[O(T^2 \cdot d) \text{ time, } O(T^2) \text{ memory}\right]$$

In contrast, **Gated Delta Net** retains an evolving state matrix $S_t \in \mathbb{R}^{d \times d}$ updated recurrently:
$$S_t = \alpha_t \odot S_{t-1}(I - \beta_t k_t k_t^\top) + \beta_t v_t k_t^\top$$
$$o_t = S_t q_t \quad \left[O(T \cdot d^2) \text{ time, } O(d^2) \text{ KV cache}\right]$$
where $\alpha_t$ acts as a decay gate and $\beta_t$ controls associative write strength.

In [ ]:
import torch
import neuralops
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt

print(f"NeuralOps Version: {neuralops.__version__}")
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 1. Inspect Architecture Spec & Mathematical Card
Before performing surgery or instantiating layers, we inspect the architectural blueprint of `gated_delta_net` through NeuralOps' canonical registry.

In [ ]:
# Inspect architectural metadata, complexity, and hardware characteristics
spec = neuralops.inspect_architecture("gated_delta_net")
card = neuralops.render_architecture_card(spec)
print(card)

## 2. Direct Layer Instantiation via `neuralops.create_attention`
NeuralOps allows instantiating any registered attention mechanism directly with a unified signature.

In [ ]:
# Instantiate standalone GatedDeltaNet layer
attn_layer = neuralops.create_attention(
    "gated_delta_net",
    embed_dim=768,
    num_heads=12,
).to(device=device, dtype=torch.float16 if device.startswith("cuda") else torch.float32)

# Verify forward pass with synthetic activations
dummy_x = torch.randn(2, 128, 768, device=device, dtype=torch.float16 if device.startswith("cuda") else torch.float32)
attn_out = attn_layer(dummy_x)
out_tensor = attn_out[0] if isinstance(attn_out, tuple) else attn_out

print(f"Layer class: {type(attn_layer).__name__}")
print(f"Input shape:  {list(dummy_x.shape)}")
print(f"Output shape: {list(out_tensor.shape)}")
assert out_tensor.shape == dummy_x.shape, "Output shape must match input shape"

## 3. Load Pretrained Host Model (GPT-2)
Next, we load a standard pretrained causal language model as our surgical host.

In [ ]:
model_name = "gpt2"
print(f"Loading host model '{model_name}'...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

num_params = sum(p.numel() for p in model.parameters())
num_layers = len(model.transformer.h)
print(f"✓ Loaded {model_name}: {num_params:,} parameters across {num_layers} transformer blocks.")

## 4. 1-Line Surgical Attention Replacement
We perform zero-friction attention replacement using `neuralops.replace_attention`. This replaces every quadratic Multi-Head Attention layer in the model with Triton-fused `gated_delta_net`, while retaining deep copies of teacher weights for calibration and distillation.

```python
neuralops.replace_attention(model, attention="gated_delta_net")
```

In [ ]:
import copy

# Maintain a bit-exact teacher copy to serve as reference ground-truth
teacher_model = copy.deepcopy(model)
teacher_model.eval()

# Execute 1-line surgical transplant across all attention blocks
neuralops.replace_attention(model, attention="gated_delta_net")
print("✓ Surgical attention replacement complete! Host attention blocks upgraded to GatedDeltaNet.")

## 5. Layerwise Subspace Micro-Healing
After replacing attention modules, representation geometry between the original teacher activations and the new student activations exhibits a distribution shift.

NeuralOps' `heal_replaced_model` solves this in two coordinated phases:
1. **Closed-Form Ridge Initialization**: Computes optimal projection weights analytically via regularized least squares ($W^* = (X^\top X + \lambda I)^{-1} X^\top Y$).
2. **Layerwise Gradient Alignment**: Minimizes representation MSE and maximizes cosine similarity:
$$\mathcal{L}_{\text{heal}} = \text{MSE}(h_{\text{student}}, h_{\text{teacher}}) - \lambda_{\text{cos}} \cdot \text{CosineSimilarity}(h_{\text{student}}, h_{\text{teacher}})$$

In [ ]:
# Calibration corpus for representation alignment
calibration_corpus = [
    "NeuralOps provides high-performance linear attention mechanisms and hardware-fused Triton kernels.",
    "Efficient transformer architectures replace quadratic dot-product attention with subquadratic recurrent states.",
    "Hardware roofline modeling characterizes arithmetic intensity, SRAM bandwidth saturation, and peak compute ceilings.",
    "Subspace alignment and closed-form Ridge initialization restore representation fidelity without full retraining."
]

calib_encodings = tokenizer(
    calibration_corpus,
    padding=True,
    truncation=True,
    return_tensors="pt"
).input_ids.to(device)

print(f"Calibration tensor shape: {list(calib_encodings.shape)} (Batch={calib_encodings.shape[0]}, Tokens={calib_encodings.shape[1]})")

# Execute micro-healing across all replaced attention blocks
healing_stats = neuralops.heal_replaced_model(
    teacher_model=teacher_model,
    student_model=model,
    calibration_inputs=calib_encodings,
    steps=15,
    lr=2e-3,
    cosine_weight=0.5,
    also_ridge_init=True,
    log_to_ledger=True,
)

print(f"✓ Micro-Healing Complete across {healing_stats['num_healed_layers']} layers in {healing_stats['elapsed_sec']:.2f}s:")
print(f"  • Global Representation MSE:    {healing_stats['initial_mse']:.4f}  →  {healing_stats['final_mse']:.4f}")
print(f"  • Global Representation CosSim: {healing_stats['initial_cosine_sim']:.4f}  →  {healing_stats['final_cosine_sim']:.4f}")

## 6. Visualization: Layerwise Convergence of Micro-Healing
To inspect how each transformer layer recovered its representation geometry, we plot initial vs. final MSE and Cosine Similarity per layer.

In [ ]:
# Extract per-layer metrics
per_layer = healing_stats.get("per_layer", {})
layers = list(range(len(per_layer)))
init_mses = [per_layer[k]["initial_mse"] for k in per_layer]
final_mses = [per_layer[k]["final_mse"] for k in per_layer]
init_coss = [per_layer[k]["initial_cosine_sim"] for k in per_layer]
final_coss = [per_layer[k]["final_cosine_sim"] for k in per_layer]

# Publication-grade visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), dpi=120)

# Subplot 1: MSE Reduction
width = 0.35
x_pos = [i for i in layers]
ax1.bar([p - width/2 for p in x_pos], init_mses, width=width, label="Pre-Healing MSE", color="#e74c3c", alpha=0.85)
ax1.bar([p + width/2 for p in x_pos], final_mses, width=width, label="Post-Healing MSE", color="#2ecc71", alpha=0.85)
ax1.set_xlabel("Transformer Block Index", fontsize=11, fontweight="bold")
ax1.set_ylabel("Representation Mean Squared Error (MSE)", fontsize=11, fontweight="bold")
ax1.set_title("Layerwise Representation Error Reduction", fontsize=12, fontweight="bold")
ax1.set_xticks(layers)
ax1.legend(frameon=True, fontsize=10)
ax1.grid(True, linestyle="--", alpha=0.5)

# Subplot 2: Cosine Similarity Alignment
ax2.plot(layers, init_coss, marker="o", linewidth=2.0, color="#e67e22", label="Pre-Healing CosSim", linestyle="--")
ax2.plot(layers, final_coss, marker="s", linewidth=2.5, color="#2980b9", label="Post-Healing CosSim")
ax2.set_xlabel("Transformer Block Index", fontsize=11, fontweight="bold")
ax2.set_ylabel("Cosine Similarity", fontsize=11, fontweight="bold")
ax2.set_title("Layerwise Subspace Directional Alignment", fontsize=12, fontweight="bold")
ax2.set_xticks(layers)
ax2.legend(frameon=True, fontsize=10)
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Save Bit-Exact Surgical Checkpoint & Query Telemetry Ledger
Once surgery and micro-healing are complete, NeuralOps persists the modified architecture along with a cryptographic surgical manifest (`surgical_manifest.json`) using `neuralops.save_surgical_checkpoint`. 

Furthermore, every operation logs execution telemetry into the local SQLite/JSONL ledger accessible via `neuralops.get_default_ledger()`.

In [ ]:
import tempfile
import os

# Save model and bit-exact surgical ledger manifest
save_dir = tempfile.mkdtemp(prefix="neuralops_checkpoint_")
saved_path = neuralops.save_surgical_checkpoint(
    model=model,
    save_dir=save_dir,
    metadata={"host_model": "gpt2", "attention_replacement": "gated_delta_net", "healed": True}
)

print(f"✓ Surgical Checkpoint successfully created at: {saved_path}")
manifest_file = os.path.join(save_dir, "surgical_manifest.json")
if os.path.exists(manifest_file):
    with open(manifest_file, "r") as f:
        manifest_head = f.read()[:350]
    print(f"✓ Surgical Manifest Verified:\n{manifest_head}...")

# Query active telemetry records from ResultsLedger
ledger = neuralops.get_default_ledger()
stats = ledger.get_summary_stats()
print(f"\nResultsLedger Summary:")
print(f"  • Total benchmarked records: {stats.get('total_records', 0)}")
print(f"  • Distinct neural modules:   {stats.get('distinct_modules', 0)}")
print(f"  • Category distribution:     {stats.get('by_category', {})}")